In [0]:
df = spark.read.format("csv") \
    .option("header", "true")\
    .option("inferSchema", "true") \
    .load("/Workspace/Users/20202406@bu.ac.kr/Test_PL_Data/test_pl_data_2.csv")
df.printSchema()

In [0]:
# 기존에 우리가 쓰던 필수 컬럼 리스트
essential_columns = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 
    'Referee', 'HS', 'AS', 'HST', 'AST', 
    'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR'
]

# 새 파일의 컬럼들과 비교해서 '빠진 컬럼'이 있는지 자동 체크
missing_columns = set(essential_columns) - set(df.columns)

if not missing_columns:
    print("✅ 검증 완료: 필수 컬럼 24개가 모두 안전하게 존재합니다! 파이프라인을 그대로 실행해도 좋습니다.")
else:
    print(f"❌ 경고: 새 파일에 다음 컬럼이 누락되었습니다: {missing_columns}")

In [0]:
display(df)

In [0]:
df.columns

In [0]:
from pyspark.sql.functions import col, to_date

# 사용할 축구 통계 컬럼만 리스트로 저장
essential_columns = [
'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam',
'FTHG', 'FTAG','FTR', 'HTHG','HTAG', 
'HTR', 'Referee', 'HS', 'AS', 'HST',
 'AST', 'HF', 'AF', 'HC', 'AC',
 'HY', 'AY', 'HR', 'AR']

### ⚽ 축구 데이터 컬럼 정의서

#### ① 기본 경기 정보
* **Div**: 리그 구분 (E0 = 잉글랜드 프리미어리그)
* **Date**: 경기 날짜 (DD/MM/YY 형식)
* **Time**: 경기 시작 시간
* **HomeTeam**: 홈팀 이름
* **AwayTeam**: 원정팀 이름

#### ② 경기 결과 및 스코어
* **FTHG**: 홈팀 최종 득점
* **FTAG**: 원정팀 최종 득점
* **FTR**: 최종 경기 결과 (H = 홈팀 승, A = 원정팀 승, D = 무승부)
* **HTHG / HTAG / HTR**: 전반전 종료 시점의 홈/원정 득점 및 결과

#### ③ 인게임 경기 통계
* **HS / AS**: 홈팀 / 원정팀의 총 슈팅 수
* **HST / AST**: 홈팀 / 원정팀의 유효 슈팅 수
* **HF / AF**: 홈팀 / 원정팀의 파울 횟수
* **HC / AC**: 홈팀 / 원정팀의 코너킥 횟수
* **HY / AY**: 홈팀 / 원정팀의 옐로카드 수
* **HR / AR**: 홈팀 / 원정팀의 레드카드 수

In [0]:
df_soccer_only = df.select(*essential_columns)

In [0]:
#3. 데이터 타입 정제 및 표준화
# 원천 데이터의 문자열을 DateType으로 파싱
df_silver = df_soccer_only.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
#4. 결과 확인
display(df_silver)

In [0]:
#5 Silver 레이어 테이블로 Delta 포맷 저장
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("premier_league_silver")

In [0]:
%sql
SELECT * FROM premier_league_silver 
ORDER BY Date DESC
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS total_matches,
    ROUND(AVG(FTHG), 2) AS avg_home_team_goals,
    ROUND(AVG(FTAG), 2) AS avg_away_away_goals,
    ROUND(AVG(HS), 2) AS avg_home_shots,
    ROUND(AVG(AS), 2) AS avg_away_shots,
    ROUND(AVG(HST), 2) AS avg_home_shots_on_target,
    ROUND(AVG(AST), 2) AS avg_away_shots_on_target
FROM premier_league_silver;


In [0]:
%sql
SELECT 
    AwayTeam AS team_name,
    COUNT(*) AS total_away_matches,                                     -- 총 원정 경기 수
    COUNT(CASE WHEN FTR = 'A' THEN 1 END) AS away_wins,                -- 원정 승리 수
    ROUND((COUNT(CASE WHEN FTR = 'A' THEN 1 END) / COUNT(*)) * 100, 1) AS away_win_pct -- 원정 승률(%)
FROM premier_league_silver
GROUP BY AwayTeam
ORDER BY away_wins DESC, away_win_pct DESC
LIMIT 10;

In [0]:
%sql
SELECT 
    AwayTeam AS team_name,
    COUNT(*) AS total_away_matches,
    -- 원정 경기 승점 계산
    SUM(CASE WHEN FTR = 'A' THEN 3 WHEN FTR = 'D' THEN 1 ELSE 0 END) AS away_points,
    COUNT(CASE WHEN FTR = 'A' THEN 1 END) AS away_wins,
    COUNT(CASE WHEN FTR = 'D' THEN 1 END) AS away_draws,
    COUNT(CASE WHEN FTR = 'H' THEN 1 END) AS away_losses,
    --원정 승이 같은 팀들의 순위 가름을 위한 자료- 원정 총 득점, 총 실점,  골득실
    SUM(FTAG) AS away_goals_scored,
    SUM(FTHG) AS away_goals_conceded,
    SUM(FTAG) - SUM(FTHG) AS away_goal_difference
FROM premier_league_silver
GROUP BY AwayTeam
ORDER BY
    away_points DESC,             -- 1순위: [기준 변경] 원정 총 승점이 높은 순
    away_goal_difference DESC,    -- 2순위: 승점 같으면 원정 골득실 높은 순
    away_goals_scored DESC,        -- 3순위: 골득실도 같으면 원정 다득점 많은 순
    away_wins DESC                -- 4순위: 다득점도 같으면 원정 승리가 많은 순
LIMIT 10;

In [0]:
%sql
SELECT 
    'Home' AS match_type,
    COUNT(*) AS matches,
    SUM(CASE WHEN FTR = 'H' THEN 3 WHEN FTR = 'D' THEN 1 ELSE 0 END) AS points,
    ROUND(AVG(FTHG), 2) AS avg_goals_scored,     -- 홈 평균 득점
    ROUND(AVG(FTAG), 2) AS avg_goals_conceded,   -- 홈 평균 실점
    ROUND(AVG(HS), 2) AS avg_shots,              -- 홈 평균 슈팅
    ROUND(AVG(HST)/AVG(HS) * 100, 1) AS shot_accuracy_pct, -- 홈 슈팅 정확도
    SUM(HY) AS total_yellow_cards,
    SUM(HR) AS total_red_cards
FROM premier_league_silver
WHERE HomeTeam = 'Tottenham'

UNION ALL

SELECT 
    'Away' AS match_type,
    COUNT(*) AS matches,
    SUM(CASE WHEN FTR = 'A' THEN 3 WHEN FTR = 'D' THEN 1 ELSE 0 END) AS points,
    ROUND(AVG(FTAG), 2) AS avg_goals_scored,     -- 원정 평균 득점
    ROUND(AVG(FTHG), 2) AS avg_goals_conceded,   -- 원정 평균 실점
    ROUND(AVG(AS), 2) AS avg_shots,              -- 원정 평균 슈팅
    ROUND(AVG(AST)/AVG(AS) * 100, 1) AS shot_accuracy_pct, -- 원정 슈팅 정확도
    SUM(AY) AS total_yellow_cards,
    SUM(AR) AS total_red_cards
FROM premier_league_silver
WHERE AwayTeam = 'Tottenham';

In [0]:
df_1617_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Workspace/Users/20202406@bu.ac.kr/Test_PL_Data/pl_1617_data_fromFDuk.csv")

In [0]:
display(df_1617_raw)

In [0]:
from pyspark.sql.functions import col, sum, when

# 각 컬럼별로 Null이 있으면 1을 주고, 그걸 다 더해서(sum) 개수를 셉니다.
null_counts = df_1617_raw.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in df_1617_raw.columns
])

display(null_counts)

In [0]:
df_1617_raw.columns

In [0]:
essential_columns = ['Div',
 'Date',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR',
 'AR']

In [0]:
from pyspark.sql.functions import col, to_date

df_1617_silver = df_1617_raw.select(*essential_columns) \
    .withColumn("Date", to_date(col("DAte"), "yyyy-MM-dd"))

In [0]:
df_1617_silver.write \
    .format("delta") \
        .mode("append") \
            .saveAsTable("premier_league_silver")

In [0]:
%sql
-- 1. 16-17 시즌 토트넘의 홈 & 원정 성적
SELECT 
    '2016-17' AS season,
    CASE WHEN HomeTeam = 'Tottenham' THEN 'Home' ELSE 'Away' END AS match_type,
    COUNT(*) AS matches,
    -- 승점 계산 (승 3점, 무 1점, 패 0점)
    SUM(CASE 
        WHEN HomeTeam = 'Tottenham' AND FTR = 'H' THEN 3 
        WHEN AwayTeam = 'Tottenham' AND FTR = 'A' THEN 3
        WHEN FTR = 'D' THEN 1 ELSE 0 END) AS points,
    -- 경기당 평균 득점 및 실점
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN FTHG ELSE FTAG END), 2) AS avg_goals_scored,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN FTAG ELSE FTHG END), 2) AS avg_goals_conceded,
    -- 경기당 평균 슈팅 및 유효 슈팅
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HS ELSE AS END), 2) AS avg_shots,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HST ELSE AST END), 2) AS avg_shots_on_target,
    -- 경기당 평균 파울 및 옐로카드 총합
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HF ELSE AF END), 2) AS avg_fouls,
    SUM(CASE WHEN HomeTeam = 'Tottenham' THEN HY ELSE AY END) AS total_yellow_cards
FROM premier_league_silver
WHERE (HomeTeam = 'Tottenham' OR AwayTeam = 'Tottenham') 
  AND Date BETWEEN '2016-08-01' AND '2017-06-01' -- 16-17 시즌 기간 필터링
GROUP BY match_type

UNION ALL

-- 2. 25-26 현재 시즌 토트넘의 홈 & 원정 성적
SELECT 
    '2025-26' AS season,
    CASE WHEN HomeTeam = 'Tottenham' THEN 'Home' ELSE 'Away' END AS match_type,
    COUNT(*) AS matches,
    SUM(CASE 
        WHEN HomeTeam = 'Tottenham' AND FTR = 'H' THEN 3 
        WHEN AwayTeam = 'Tottenham' AND FTR = 'A' THEN 3
        WHEN FTR = 'D' THEN 1 ELSE 0 END) AS points,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN FTHG ELSE FTAG END), 2) AS avg_goals_scored,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN FTAG ELSE FTHG END), 2) AS avg_goals_conceded,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HS ELSE AS END), 2) AS avg_shots,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HST ELSE AST END), 2) AS avg_shots_on_target,
    ROUND(AVG(CASE WHEN HomeTeam = 'Tottenham' THEN HF ELSE AF END), 2) AS avg_fouls,
    SUM(CASE WHEN HomeTeam = 'Tottenham' THEN HY ELSE AY END) AS total_yellow_cards
FROM premier_league_silver
WHERE (HomeTeam = 'Tottenham' OR AwayTeam = 'Tottenham') 
  AND Date BETWEEN '2025-08-01' AND '2026-05-25' -- 25-26 시즌 기간 필터링
GROUP BY match_type
ORDER BY season DESC, match_type DESC;

In [0]:
%sql
-- 토트넘의 홈 경기만 집중 분석 (16-17 vs 25-26)
SELECT 
    CASE WHEN YEAR(Date) IN (2016, 2017) THEN '2016-17 (White Hart Lane)' 
         ELSE '2025-26 (Tottenham Hotspur Stadium)' END AS stadium_era,
    COUNT(*) AS home_matches,
    
    -- 1. 공격 인프라 지표 (우리팀 코너킥)
    ROUND(AVG(HC), 2) AS avg_tottenham_corners,
    
    -- 2. 수비 인프라 지표 (상대팀이 때린 슛 & 유효슛)
    ROUND(AVG(AS), 2) AS avg_opponent_shots,
    ROUND(AVG(AST), 2) AS avg_opponent_shots_on_target,
    
    -- 3. 거친 수비 지표 (파울 10개당 옐로카드 비율)
    SUM(HY) AS total_tottenham_yellows,
    ROUND((SUM(HY) / SUM(HF)) * 10, 2) AS cards_per_10_fouls
    
FROM premier_league_silver
WHERE HomeTeam = 'Tottenham' 
  AND ((YEAR(Date) = 2016 OR YEAR(Date) = 2017) OR (YEAR(Date) = 2025 OR YEAR(Date) = 2026))
GROUP BY stadium_era;